# Market Basket Analysis Using Association Rule Mining
**Author:** Zaldy D. Biliran
Radiant WinstonRyan Soberano
Jhonette Tiara Jimenea (BSIS 4A)  
**Instructor:** Joyce F. Jamile  
  

**Project Summary:** This project applies the Apriori Association Rule Mining algorithm to 541,909 historical e-commerce transaction records from the UCI Online Retail dataset to discover products frequently bought together. The discovered rules are evaluated using Support, Confidence, and Lift against a statistical independence baseline (Lift = 1.00) to support product bundling, cross-selling, and inventory planning.

---
## A. Problem Definition

### Brief Description
Businesses collect large volumes of historical retail transaction data, yet raw receipt logs alone do not show which items customers frequently buy together. This notebook applies **Market Basket Analysis** using **Association Rule Mining (Apriori algorithm)** to uncover meaningful product co-occurrence patterns from the Online Retail Excel dataset (`Online Retail.xlsx`).

### Problem Statement
Using 541,909 historical e-commerce transaction line items from the UCI Online Retail dataset (December 2010 to December 2011), this project applies the Apriori Association Rule Mining algorithm to identify frequent product combinations and association rules that outperform a random guessing baseline (Lift greater than 1.0), enabling businesses to optimize product placement, promotions, bundling, and cross-selling.

 1. Required Packages and Reproducibility Setup
**What we are doing and why:** We import the required data manipulation (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), and Association Rule Mining (`mlxtend`) libraries, and set a fixed random seed (`42`) so all results are fully reproducible.

**Output Interpretation:** All required libraries are loaded cleanly and the random seed is locked to `42` to guarantee consistent train/test splitting and evaluation.

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Random seed set to: {RANDOM_SEED}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Random seed set to: 42
pandas version: 2.2.3
numpy version: 2.4.6


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


2. Dataset Loading and Initial Inspection
**What we are doing and why:** We load the `Online Retail.xlsx` dataset using a relative file path (keeping the notebook portable across computers) and inspect its shape, column names, and first 5 rows to verify that the data loaded properly.

**Output Interpretation:** The dataset loads with **541,909 rows and 8 columns** (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`), confirming that it satisfies the project requirement of at least 500 rows and 5 usable features and is ready for **Section B (Summarize and Analyze Data)**.

In [6]:
!pip install -q openpyxl mlxtend

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import os
import urllib.request
import zipfile
import pandas as pd
from google.colab import drive

# 1. Connect Google Colab to your Google Drive
drive.mount('/content/drive')

# 2. Point to your Google Drive folder
FOLDER_PATH = "/content/drive/MyDrive/FACILIX MACHINE LEARNING PROJECT"
os.makedirs(FOLDER_PATH, exist_ok=True)
DATA_PATH = os.path.join(FOLDER_PATH, "Online Retail.xlsx")

# 3. Download from UCI only if not already in Drive
if not os.path.exists(DATA_PATH):
    print("Downloading Online Retail.xlsx from UCI...")
    zip_path = "/content/online_retail.zip"
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/static/public/352/online+retail.zip",
        zip_path
    )
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(FOLDER_PATH)
    print("Saved 'Online Retail.xlsx' into your Google Drive folder!")
else:
    print("Found existing file in Drive:", DATA_PATH)

# 4. Load and display the first 5 rows
df = pd.read_excel(DATA_PATH, engine="openpyxl")
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found existing file in Drive: /content/drive/MyDrive/FACILIX MACHINE LEARNING PROJECT/Online Retail.xlsx
Dataset Shape: 541,909 rows x 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# **A2**. Is this classification, regression, clustering or something else? Why does that framing fit the problem?



**ANSWER**: **Unsupervised Association Rule Mining (Pattern Mining)** — falls under the "something else" category. Our dataset is a collection of customer shopping receipts with no single target label to predict (not classification or regression) and no fixed customer clusters to partition (not clustering). Instead, Association Rule Mining scans thousands of receipts to find "if-then" co-occurrence patterns between products bought in the same basket.

# **A3**. What is your target or outcome of interest and how was it measured or labelled?

**ANSWER:** Because Association Rule Mining is unsupervised, there is no single target column in the raw file. The outcome of interest is a set of **frequent itemsets and directional association rules** (linking an "if" antecedent product set to a "then" consequent product set). Raw transaction rows are cleaned (removing cancelled invoices and non-positive quantities/prices), grouped by `InvoiceNo`, and converted into a binary basket table (`1` if purchased in that invoice, `0` if not).

# **A4**. Which metric(s) define success, and why do they match the real cost of different errors?

**ANSWER:**

Three metrics define success:<br>1. **Support:** How often two items are bought together out of all receipts.<br>2. **Confidence:** How often customers buy the second item when they already have the first item.<br>3. **Lift:** How much more likely the second item is bought with the first item compared to random coincidence (`Lift > 1.0` = positive link; `Lift = 1.0` = pure luck).<br>**Matching Real Error Costs:** Minimum **Support** prevents wasting marketing money on rare combos almost nobody buys, while requiring **Lift > 1.0** prevents giving away unnecessary bundle discounts on popular items customers would buy at full price anyway.


# **A7**. What score would a naive baseline achieve?

**ANSWER:**

 The naive baseline is **Random Guessing (Statistical Independence)** — assuming what a customer puts in their basket has zero effect on what else they buy.<br>• **Naive Baseline Lift = `1.00`** (buying item A does not change the normal purchase rate of item B).<br>• **Naive Baseline Confidence:** Suggesting the single most popular store item to every shopper achieves Confidence equal to that top item's standalone Support (~8%–11%), with **Lift = `1.00`**. Any useful Apriori rule must beat `Lift = 1.00` on unseen test receipts. |






In [5]:

# Final Cell: Print Package Versions & Generate requirements.txt
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import openpyxl
import mlxtend

print(f"Python: {sys.version.split()[0]}")
packages = [pd, np, matplotlib, sns, openpyxl, mlxtend]

with open("requirements.txt", "w") as f:
    for pkg in packages:
        line = f"{pkg.__name__}=={pkg.__version__}"
        print(line)
        f.write(line + "\n")

print("\nrequirements.txt created successfully!")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Python: 3.13.18
pandas==2.2.3
numpy==2.4.6
matplotlib==3.10.0
seaborn==0.13.2
openpyxl==3.1.5
mlxtend==0.23.4

requirements.txt created successfully!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag